In [0]:
from pyspark.sql import functions as F

CATALOG = "github_pipeline_catalog"
SCHEMA = "github_analytics"

In [0]:
repo_dim = spark.table(
    f"{CATALOG}.{SCHEMA}."
    "gold_dim_repository"
)

assert (
    repo_dim.count() == repo_dim.select("repo_id").distinct().count()
)

print(
    "PASS: repository dimension "
    "has unique repo_id"
)

In [0]:
snapshot = spark.table(
    f"{CATALOG}.{SCHEMA}."
    "gold_fact_repository_snapshot"
)

total = snapshot.count()

unique = (
    snapshot.select("repo_id", "date_key").distinct().count()
)

print("Snapshot total:", total)
print("Snapshot unique keys:", unique)

assert total == unique

print("PASS: snapshot fact PK")

In [0]:
activity = spark.table(
    f"{CATALOG}.{SCHEMA}."
    "gold_fact_repository_activity"
)

total = activity.count()

unique = (
    activity
    .select(
        "repo_id",
        "date_key"
    )
    .distinct()
    .count()
)

assert total == unique

print("PASS: activity fact PK")

In [0]:
orphan_repo_activity = (
    activity.alias("f")
    .join(
        repo_dim.alias("d"),
        F.col("f.repo_id")
        ==
        F.col("d.repo_id"),
        "left_anti"
    )
    .count()
)

assert orphan_repo_activity == 0

print("PASS: activity repo FK")

In [0]:
dim_date = spark.table(
    f"{CATALOG}.{SCHEMA}."
    "gold_dim_date"
)

orphan_date_activity = (
    activity.alias("f")
    .join(
        dim_date.alias("d"),
        F.col("f.date_key")
        ==
        F.col("d.date_key"),
        "left_anti"
    )
    .count()
)

assert orphan_date_activity == 0

print(
    "PASS: activity date FK"
)

In [0]:
tables = [
    "gold_dim_repository",
    "gold_fact_repository_snapshot",
    "gold_fact_repository_activity",
    "gold_fact_pipeline_execution"
]

for table in tables:

    missing = (
        spark.table(
            f"{CATALOG}.{SCHEMA}.{table}"
        )
        .filter(
            F.col(
                "load_timestamp"
            ).isNull()
        )
        .count()
    )

    print(
        table,
        "NULL load_timestamp:",
        missing
    )

In [0]:
display(
    spark.sql(f"""
        SELECT
            layer,
            dataset,
            status,
            rows_read,
            rows_inserted,
            rows_updated,
            execution_start,
            execution_end
        FROM
            `{CATALOG}`.`{SCHEMA}`.
            `pipeline_execution_logs`
        WHERE
            layer = 'Silver-to-Gold'
        ORDER BY
            execution_start DESC
    """)
)